# DS2 · Experimentación avanzada — Práctica guiada

**Caso TiendaSur: rediseño del carrito.** Producto quiere lanzar un carrito nuevo. Tú diseñas y lees el experimento: cuánta gente hace falta, cómo ganar sensibilidad sin más tráfico (CUPED), por qué no se puede «mirar a ver cómo va», qué hacer con 20 métricas, cómo detectar un experimento roto (SRM), qué pasa con el efecto novedad y las interferencias, y cuándo conviene un *bandit* en vez de un A/B.

Todo se simula, así que conocemos la verdad y podemos medir si cada técnica funciona. CPU, pocos minutos, sin clave de API.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.3f}".format
rng = np.random.default_rng(2026)

def dif_medias(y_t, y_c):
    """Diferencia de medias (tratamiento - control), error estándar, IC 95 % y p-valor (aprox. normal)."""
    y_t, y_c = np.asarray(y_t, float), np.asarray(y_c, float)
    d = y_t.mean() - y_c.mean()
    se = np.sqrt(y_t.var(ddof=1) / len(y_t) + y_c.var(ddof=1) / len(y_c))
    p = 2 * (1 - stats.norm.cdf(abs(d / se)))
    return d, se, (d - 1.96 * se, d + 1.96 * se), p

print("Entorno listo")

## DS2.1 · Repaso: potencia y tamaño de muestra

Cuatro cantidades se determinan entre sí: **α** (falsos positivos tolerados), **potencia** (1 − β: probabilidad de detectar un efecto real), el **efecto mínimo detectable** (MDE) y el **tamaño de muestra**. Fijadas tres, sale la cuarta. Lo importante para negocio: *detectar mejoras pequeñas cuesta muchísimo tráfico* (n crece con 1/MDE²).

In [ ]:
# Caso TiendaSur: rediseño del carrito. Métrica principal: conversión de sesión con carrito a compra.
P0 = 0.10                                   # conversión actual
def n_por_grupo(p0, mde_relativo, alfa=0.05, potencia=0.80):
    """Tamaño de muestra por grupo para detectar una mejora relativa `mde_relativo` (test bilateral)."""
    h = proportion_effectsize(p0 * (1 + mde_relativo), p0)       # tamaño del efecto de Cohen (h)
    return NormalIndPower().solve_power(h, alpha=alfa, power=potencia, ratio=1.0, alternative="two-sided")

filas = [{"mejora relativa a detectar": f"{m:.0%}", "conversión nueva": P0 * (1 + m), "n por grupo": n_por_grupo(P0, m)}
         for m in [0.02, 0.05, 0.10, 0.20]]
tabla = pd.DataFrame(filas)
tabla["n por grupo"] = tabla["n por grupo"].round(0).astype(int)
print(tabla.to_string(index=False))

# ¿Se cumple lo prometido? Simulamos 5 000 experimentos con la mejora del 10 % y miramos cuántos salen significativos
n = int(n_por_grupo(P0, 0.10))
reps = 5000
x_c = rng.binomial(n, P0, reps); x_t = rng.binomial(n, P0 * 1.10, reps)
p_hat_c, p_hat_t = x_c / n, x_t / n
p_comun = (x_c + x_t) / (2 * n)
z = (p_hat_t - p_hat_c) / np.sqrt(2 * p_comun * (1 - p_comun) / n)
potencia_emp = np.mean(np.abs(z) > 1.96)
# Y si NO hay efecto real (A/A): ¿cuántos falsos positivos?
x_t0 = rng.binomial(n, P0, reps)
p0_comun = (x_c + x_t0) / (2 * n)
z0 = (x_t0 / n - p_hat_c) / np.sqrt(2 * p0_comun * (1 - p0_comun) / n)
print(f"\nn = {n:,} por grupo | potencia empírica con efecto real = {potencia_emp:.1%} (objetivo 80 %) | falsos positivos sin efecto = {np.mean(np.abs(z0) > 1.96):.1%} (objetivo 5 %)")

# Con una métrica continua (gasto por sesión): n depende de la desviación típica
sigma, mde_abs = 40.0, 1.0       # σ = 40 €, queremos detectar 1 €
n_cont = TTestIndPower().solve_power(effect_size=mde_abs / sigma, alpha=0.05, power=0.8)
print(f"Métrica continua: σ = {sigma:.0f} €, efecto mínimo = {mde_abs:.0f} € -> {n_cont:,.0f} por grupo")

## DS2.2 · CUPED: más sensibilidad con los mismos usuarios

**CUPED** (*Controlled-experiment Using Pre-Experiment Data*) usa una covariable medida **antes** del experimento (por ejemplo el gasto de la semana anterior) para restar la parte de la variación que ya sabíamos que ese usuario tendría. Como el tratamiento no puede haber cambiado algo anterior al experimento, el ajuste **no introduce sesgo**, y la varianza baja un factor (1 − ρ²).

In [ ]:
def generar_usuarios(n, efecto=1.0, rho_objetivo=0.7, rng=None):
    """Gasto semanal (€) de n usuarios por grupo. X = gasto de la semana ANTERIOR al experimento (no lo toca el tratamiento).
    El nivel propio de cada usuario hace que X e Y estén correlacionados."""
    rng = rng or np.random.default_rng(0)
    var_nivel = 900.0
    var_ruido = var_nivel * (1 - rho_objetivo) / rho_objetivo
    d = {}
    for g, T in [("control", 0), ("tratamiento", 1)]:
        nivel = rng.normal(50, np.sqrt(var_nivel), n)
        d[g] = pd.DataFrame({"grupo": g, "x_pre": nivel + rng.normal(0, np.sqrt(var_ruido), n),
                             "y": nivel + efecto * T + rng.normal(0, np.sqrt(var_ruido), n)})
    return pd.concat(d.values(), ignore_index=True)

def cuped(y, x, grupo):
    """Ajuste CUPED: y' = y - θ·(x - media(x)), con θ = cov(y, x) / var(x) estimado con ambos grupos."""
    theta = np.cov(y, x)[0, 1] / x.var(ddof=1)
    return y - theta * (x - x.mean()), theta

u = generar_usuarios(10_000)
rho = u[["x_pre", "y"]].corr().iloc[0, 1]
u["y_cuped"], theta = cuped(u["y"].to_numpy(), u["x_pre"].to_numpy(), u["grupo"])
for etiqueta, col in [("Sin CUPED", "y"), ("Con CUPED", "y_cuped")]:
    d, se, ic, p = dif_medias(u.loc[u.grupo == "tratamiento", col], u.loc[u.grupo == "control", col])
    print(f"{etiqueta:10s} efecto = {d:5.2f} € | error estándar = {se:.3f} | IC95% [{ic[0]:5.2f}, {ic[1]:5.2f}] | p = {p:.4f}")
print(f"\ncorrelación pre/post ρ = {rho:.2f} | θ = {theta:.2f} | reducción teórica de varianza 1 - ρ² = {1 - rho**2:.2f}")
print(f"varianza de y: {u['y'].var():.0f} -> {u['y_cuped'].var():.0f}  (ratio {u['y_cuped'].var() / u['y'].var():.2f})")

In [ ]:
# Repetimos el experimento 2 000 veces (n = 1 000 por grupo) con un efecto real de 1 €: ¿es insesgado? ¿cuánto baja la dispersión?
def simular_cuped(rho, reps=2000, n=1000, efecto=1.0, semilla=1):
    r = np.random.default_rng(semilla)
    var_nivel = 900.0
    var_ruido = var_nivel * (1 - rho) / rho
    nivel = r.normal(50, np.sqrt(var_nivel), (reps, 2 * n))
    x = nivel + r.normal(0, np.sqrt(var_ruido), (reps, 2 * n))
    T = np.r_[np.zeros(n), np.ones(n)]
    y = nivel + efecto * T + r.normal(0, np.sqrt(var_ruido), (reps, 2 * n))
    xc = x - x.mean(axis=1, keepdims=True)
    theta = (xc * (y - y.mean(axis=1, keepdims=True))).sum(1) / (xc ** 2).sum(1)
    yc = y - theta[:, None] * xc
    ef = lambda m: m[:, n:].mean(1) - m[:, :n].mean(1)
    return ef(y), ef(yc)

res = []
for rho in [0.05, 0.3, 0.5, 0.7, 0.9]:
    e_normal, e_cuped = simular_cuped(rho)
    res.append({"ρ pre/post": rho, "media sin CUPED": e_normal.mean(), "media con CUPED": e_cuped.mean(),
                "desv. sin CUPED": e_normal.std(), "desv. con CUPED": e_cuped.std(),
                "varianza que queda": (e_cuped.std() / e_normal.std()) ** 2, "teoría 1-ρ²": 1 - rho ** 2})
print(pd.DataFrame(res).to_string(index=False))
print("\n(Efecto real = 1,00 €: las medias deben rondarlo con y sin CUPED -> el ajuste NO introduce sesgo.)")

**Lee la tabla.** Con ρ = 0,7 la varianza se reduce a ≈ 51 % del original: equivale a duplicar el tráfico. Con ρ ≈ 0 no hay ganancia (y tampoco daño). En la práctica CUPED rinde mucho en métricas persistentes (gasto, sesiones) y poco en métricas de eventos raros.

## DS2.3 · Mirar antes de tiempo: el problema y sus soluciones

Un panel de experimento que se actualiza a diario tienta a parar «en cuanto sale significativo». Cada mirada es una nueva oportunidad de equivocarse, y el error de tipo I se acumula. Hay dos familias de soluciones:
- **Diseños de grupos secuenciales** con *alpha spending*: se fijan de antemano K miradas y un límite para cada una (Pocock: constante; O'Brien-Fleming: muy exigente al principio y casi 1,96 al final).
- **SPRT** y variantes (mSPRT, valores-p «siempre válidos»): se puede mirar continuamente.

Primero calibramos los límites por simulación (didáctico; en producción usa `gsDesign`, `rpact` o la librería de tu plataforma).

In [ ]:
K = 10                      # número de miradas, a intervalos iguales de datos
def trayectorias_z(reps, K, rng, deriva=0.0):
    """z-estadísticos en las K miradas. Con incrementos N(deriva, 1) por bloque: S_k = Σ, z_k = S_k / √k."""
    inc = rng.normal(deriva, 1.0, (reps, K))
    return np.cumsum(inc, axis=1) / np.sqrt(np.arange(1, K + 1))

# Calibración por simulación de dos límites clásicos (en producción se usan librerías como gsDesign o rpact):
cal = trayectorias_z(300_000, K, np.random.default_rng(42))
c_pocock = np.quantile(np.abs(cal).max(axis=1), 0.95)                                            # límite constante
c_obf = np.quantile((np.abs(cal) * np.sqrt(np.arange(1, K + 1) / K)).max(axis=1), 0.95)         # estilo O'Brien-Fleming
k = np.arange(1, K + 1)
limites = pd.DataFrame({"mirada": k, "ingenuo (1,96)": 1.96, "Pocock": c_pocock, "O'Brien-Fleming": c_obf * np.sqrt(K / k)})
print(limites.round(2).to_string(index=False))

In [ ]:
# 1) A/A (sin efecto real): ¿cuántas veces declararía «significativo» alguien que mira 10 veces y para al primer p < 0,05?
def evaluar(deriva, reps=20_000, semilla=7):
    z = trayectorias_z(reps, K, np.random.default_rng(semilla), deriva)
    reglas = {"Un solo análisis al final": np.full(K, np.inf), "Mirar 10 veces con 1,96": np.full(K, 1.96),
              "Pocock": np.full(K, c_pocock), "O'Brien-Fleming": c_obf * np.sqrt(K / k)}
    reglas["Un solo análisis al final"][-1] = 1.96
    out = {}
    for nombre, lim in reglas.items():
        cruza = np.abs(z) > lim
        alguna = cruza.any(axis=1)
        primera = np.where(alguna, cruza.argmax(axis=1) + 1, K)
        out[nombre] = {"% declara efecto": alguna.mean(), "mirada media de parada": primera.mean()}
    return pd.DataFrame(out).T

print("SIN efecto real (cada «efecto» declarado es un falso positivo):")
print((evaluar(0.0) * [100, 1]).round(1).to_string())
print("\nCON efecto real (deriva 0,8 por bloque -> z final ≈ 2,5):")
print((evaluar(0.8) * [100, 1]).round(1).to_string())

**Lee la tabla.** Mirar 10 veces con 1,96 dispara los falsos positivos hasta ≈ 20 %: uno de cada cinco «ganadores» sería humo. Pocock y O'Brien-Fleming mantienen el 5 % global. O'Brien-Fleming conserva casi toda la potencia del análisis único y es la opción habitual; Pocock permite parar antes cuando el efecto es grande, a costa de algo de potencia al final.

In [ ]:
# SPRT: test secuencial de razón de verosimilitudes. Comparamos H0: δ = 0 contra H1: δ = δ1 (efecto mínimo que nos importa).
# Datos en pares (un usuario de cada grupo, σ = 1): d_i ~ N(δ, 2). Tras cada par se suma el logaritmo del cociente de verosimilitudes.
ALFA, BETA, DELTA1 = 0.05, 0.20, 0.05
A_sup = np.log((1 - BETA) / ALFA)      # cruzar por arriba -> aceptamos H1
B_inf = np.log(BETA / (1 - ALFA))      # cruzar por abajo  -> aceptamos H0

def simular_sprt(delta_real, reps=1500, max_pares=40_000, bloque=500, semilla=3):
    r = np.random.default_rng(semilla)
    llr = np.zeros(reps); parada = np.zeros(reps, int); decision = np.zeros(reps, int)   # 0 sin decidir, 1 H1, -1 H0
    vivos = np.ones(reps, bool)
    for ini in range(0, max_pares, bloque):
        d = r.normal(delta_real, np.sqrt(2), (reps, bloque))
        inc = np.cumsum((DELTA1 / 2) * (d - DELTA1 / 2), axis=1) + llr[:, None]
        sup, inf = inc >= A_sup, inc <= B_inf
        cruza = (sup | inf) & vivos[:, None]
        hay = cruza.any(axis=1)
        pos = cruza.argmax(axis=1)
        nuevo = hay & vivos
        parada[nuevo] = ini + pos[nuevo] + 1
        decision[nuevo] = np.where(sup[np.arange(reps), pos][nuevo], 1, -1)
        vivos &= ~hay
        llr = inc[:, -1]
        if not vivos.any():
            break
    parada[vivos] = max_pares
    return decision, parada

n_fijo = TTestIndPower().solve_power(effect_size=DELTA1, alpha=ALFA, power=1 - BETA)
for nombre, dr in [("H0 cierta (δ = 0)", 0.0), ("H1 cierta (δ = 0,05)", DELTA1)]:
    dec, par = simular_sprt(dr)
    print(f"{nombre:22s} acepta H1: {np.mean(dec == 1):5.1%} | acepta H0: {np.mean(dec == -1):5.1%} | sin decidir: {np.mean(dec == 0):4.1%} | "
          f"pares medios: {par.mean():,.0f}")
print(f"Diseño de tamaño fijo equivalente: {n_fijo:,.0f} por grupo")
print("-> Con la misma tasa de falsos positivos (α) el SPRT para antes de media; pero la mirada es continua y el tamaño no está garantizado.")

## DS2.4 · Comparaciones múltiples

Si miras 20 métricas con α = 0,05, es casi seguro que alguna salga «significativa» por azar. Dos filosofías: controlar la probabilidad de **algún** falso positivo (Bonferroni, Holm: *FWER*) o la **proporción** de falsos entre los descubrimientos (Benjamini-Hochberg: *FDR*). En experimentación se suele declarar **una** métrica principal (sin corrección) y corregir el resto.

In [ ]:
# 20 métricas del experimento del carrito, ninguna con efecto real: ¿cuántas veces sale algo «significativo»?
M, EXPS = 20, 5000
z_nulo = np.random.default_rng(5).normal(0, 1, (EXPS, M))
p_nulo = 2 * (1 - stats.norm.cdf(np.abs(z_nulo)))
print(f"P(al menos una métrica con p < 0,05 | todo nulo) simulado = {np.mean((p_nulo < 0.05).any(axis=1)):.1%} | teórico 1-0,95^20 = {1 - 0.95 ** M:.1%}")

def rechazos_holm(p, alfa=0.05):
    """Holm por filas (vectorizado): el i-ésimo menor p se compara con α/(m-i); se para en el primer fallo."""
    m = p.shape[1]
    orden = np.argsort(p, axis=1)
    ordenado = np.take_along_axis(p, orden, axis=1)
    pasa = np.cumprod(ordenado <= alfa / (m - np.arange(m)), axis=1).astype(bool)
    out = np.zeros_like(pasa)
    np.put_along_axis(out, orden, pasa, axis=1)
    return out

def rechazos_bh(p, alfa=0.05):
    """Benjamini-Hochberg por filas: se rechazan todos los p <= p_(k), con k el mayor tal que p_(k) <= k/m·α."""
    m = p.shape[1]
    orden = np.argsort(p, axis=1)
    ordenado = np.take_along_axis(p, orden, axis=1)
    cumple = ordenado <= (np.arange(1, m + 1) / m) * alfa
    k = np.where(cumple.any(axis=1), m - np.argmax(cumple[:, ::-1], axis=1), 0)
    pasa = np.arange(m) < k[:, None]
    out = np.zeros_like(pasa)
    np.put_along_axis(out, orden, pasa, axis=1)
    return out

# Ahora 4 métricas con efecto real (z medio = 3) y 16 sin efecto
def comparar(reps=3000, reales=4, mu=3.0, semilla=9):
    r = np.random.default_rng(semilla)
    mus = np.r_[np.full(reales, mu), np.zeros(M - reales)]
    z = r.normal(mus, 1, (reps, M))
    p = 2 * (1 - stats.norm.cdf(np.abs(z)))
    filas = {}
    reglas = {"Sin corregir": lambda p: p < 0.05, "Bonferroni": lambda p: p < 0.05 / M,
              "Holm": rechazos_holm, "Benjamini-Hochberg": rechazos_bh}
    for nombre, regla in reglas.items():
        rechaza = regla(p)
        verdaderos = rechaza[:, :reales].sum(1)
        falsos = rechaza[:, reales:].sum(1)
        total = np.maximum(verdaderos + falsos, 1)
        filas[nombre] = {"efectos reales detectados": verdaderos.mean() / reales,
                         "falsos positivos por experimento": falsos.mean(),
                         "P(≥1 falso positivo)": (falsos > 0).mean(),
                         "proporción de falsos entre descubrimientos": (falsos / total).mean()}
    return pd.DataFrame(filas).T
comparar()

## DS2.5 · Guardarraíles y *sample ratio mismatch* (SRM)

El **SRM** ocurre cuando el reparto real entre grupos no coincide con el diseñado (p. ej. 50/50). Es la alarma más barata y más valiosa: indica que algo roto (redirecciones, bots, errores de registro) hace que los grupos ya no sean comparables, y entonces **ningún resultado es fiable**, aunque salga p = 0,001.

In [ ]:
def srm(n_t, n_c, prop_t=0.5):
    """Test chi-cuadrado de bondad de ajuste: ¿el reparto observado es compatible con el diseñado?"""
    n = n_t + n_c
    esperado = [n * prop_t, n * (1 - prop_t)]
    chi2, p = stats.chisquare([n_t, n_c], esperado)
    return p

for nombre, nt, nc in [("Experimento A", 50_312, 49_688), ("Experimento B", 50_000, 48_800), ("Experimento C", 25_400, 25_000)]:
    p = srm(nt, nc)
    print(f"{nombre}: {nt:,} / {nc:,}  ({nt / (nt + nc):.2%} en tratamiento) -> p = {p:.5f}  {'<<< SRM: NO te fíes del resultado' if p < 0.001 else 'ok'}")

# ¿Por qué importa? El efecto verdadero es CERO, pero una versión del carrito falla en móviles lentos (usuarios de bajo gasto)
r = np.random.default_rng(13)
n = 50_000
lento = r.random(2 * n) < 0.12                         # 12 % usan un móvil lento y gastan menos
gasto_base = r.lognormal(3.0, 0.8, 2 * n) * np.where(lento, 0.4, 1.0)
tratado = np.r_[np.zeros(n), np.ones(n)].astype(bool)
y = gasto_base + r.normal(0, 3, 2 * n)                 # sin efecto del tratamiento
se_pierde = tratado & lento & (r.random(2 * n) < 0.9)  # en tratamiento, el 90 % de los lentos NO llega a registrarse
y_obs, t_obs = y[~se_pierde], tratado[~se_pierde]
d, se, ic, p = dif_medias(y_obs[t_obs], y_obs[~t_obs])
print(f"\nEfecto real = 0 € | efecto estimado con los datos que sobreviven = {d:+.2f} € (IC95% [{ic[0]:.2f}, {ic[1]:.2f}], p = {p:.4f})")
print(f"Reparto observado: {t_obs.sum():,} vs {(~t_obs).sum():,} -> SRM p = {srm(int(t_obs.sum()), int((~t_obs).sum())):.2e}")

**Guardarraíles.** Métricas que no deben empeorar (latencia, devoluciones, quejas, ingresos a largo plazo). Se evalúan con **no inferioridad**: no basta con «no hay diferencia significativa» (falta de potencia ≠ ausencia de daño); se exige que el límite del IC deje fuera un empeoramiento mayor que un margen acordado de antemano.

In [ ]:
def guardarrail(y_t, y_c, margen, mayor_es_peor=True):
    """Test de no inferioridad: ¿podemos descartar un empeoramiento mayor que `margen`?
    Se mira el extremo del IC unilateral 95 %, no el p-valor de 'hay diferencia'."""
    d, se, _, _ = dif_medias(y_t, y_c)
    if mayor_es_peor:
        limite = d + 1.645 * se
        ok = limite < margen
    else:
        limite = d - 1.645 * se
        ok = limite > -margen
    return d, limite, ok

r = np.random.default_rng(17)
n = 40_000
carga_c = r.gamma(8, 100, n)                       # ms, media ≈ 800
for etiqueta, empeora in [("rediseño ligero (+20 ms reales)", 20), ("rediseño pesado (+45 ms reales)", 45)]:
    carga_t = r.gamma(8, 100, n) + empeora
    d, lim, ok = guardarrail(carga_t, carga_c, margen=30)
    print(f"{etiqueta}: dif = {d:5.1f} ms, límite superior 95 % = {lim:5.1f} ms (margen tolerado 30 ms) -> {'PASA' if ok else 'FALLA el guardarraíl'}")

print("\nMatriz de decisión:")
print(pd.DataFrame({"métrica principal": ["mejora", "mejora", "sin cambio", "mejora"],
                    "guardarraíles": ["OK", "fallan", "OK", "SRM / calidad de datos"],
                    "decisión": ["lanzar", "no lanzar o iterar: el coste oculto anula la mejora", "no lanzar (o decidir por coste/simplicidad)", "investigar, no decidir"]}).to_string(index=False))

## DS2.6 · Efecto novedad y efectos de red

**Novedad:** la gente prueba lo nuevo por curiosidad y la mejora se desinfla. Si lanzas por la primera semana, sobreestimas. **Interferencia:** el supuesto básico del A/B es que el resultado de un usuario no depende del grupo de los demás (SUTVA). En mercados, redes sociales o logística compartida no se cumple, y aleatorizar por usuario da un efecto que **no** coincide con el del lanzamiento completo.

In [ ]:
# Efecto novedad: el carrito llama la atención al principio y el interés se apaga. Efecto estable = +3 %, novedad = +9 % que decae en ~5 días.
r = np.random.default_rng(23)
DIAS, N_DIA, P0 = 28, 40_000, 0.10
estable, novedad, tau = 0.03, 0.09, 5.0
filas = []
for dia in range(DIAS):
    lift = estable + novedad * np.exp(-dia / tau)
    xc = r.binomial(N_DIA, P0); xt = r.binomial(N_DIA, P0 * (1 + lift))
    filas.append({"dia": dia + 1, "lift_real": lift, "lift_obs": (xt / xc) - 1})
nov = pd.DataFrame(filas)
nov["semana"] = (nov["dia"] - 1) // 7 + 1
print(nov.groupby("semana")[["lift_real", "lift_obs"]].mean().rename(columns={"lift_real": "lift real medio", "lift_obs": "lift observado"}).map(lambda v: f"{v:+.1%}"))
fig, ax = plt.subplots()
ax.plot(nov["dia"], nov["lift_obs"], "o-", ms=3, label="lift observado (diario)")
ax.plot(nov["dia"], nov["lift_real"], "r--", label="lift real")
ax.axhline(estable, color="grey", lw=.8, ls=":"); ax.axhline(0, color="k", lw=.5)
ax.set(xlabel="día del experimento", ylabel="mejora relativa de la conversión", title="Efecto novedad"); ax.legend(); plt.show()
print(f"Si paras a los 7 días estimas ≈ {nov.query('dia<=7').lift_obs.mean():+.1%}; el efecto a largo plazo es {estable:+.1%}. Mide la última semana, no la media.")

In [ ]:
# Interferencia: 400 zonas de reparto con 50 usuarios cada una. Los usuarios con el nuevo flujo «roban» pedidos a los demás de su zona
# (recursos compartidos: repartidores, stock). Efecto directo τ = +1,0; competencia γ = 0,8 por fracción tratada en la zona.
# Efecto del LANZAMIENTO COMPLETO (todos tratados) = τ - γ = +0,2.
TAU, GAMMA, ZONAS, USUARIOS = 1.0, 0.8, 400, 50

def experimento(nivel, semilla):
    r = np.random.default_rng(semilla)
    zona = np.repeat(np.arange(ZONAS), USUARIOS)
    if nivel == "usuario":
        t = r.binomial(1, 0.5, ZONAS * USUARIOS)
    else:
        t = np.repeat(r.permutation(np.r_[np.ones(ZONAS // 2), np.zeros(ZONAS // 2)]), USUARIOS).astype(int)
    frac = pd.Series(t).groupby(zona).transform("mean").to_numpy()
    y = 10 + TAU * t - GAMMA * frac + r.normal(0, 3, len(t))
    if nivel == "usuario":
        return y[t == 1].mean() - y[t == 0].mean()
    por_zona = pd.DataFrame({"zona": zona, "t": t, "y": y}).groupby("zona").agg(t=("t", "first"), y=("y", "mean"))
    return por_zona.query("t==1").y.mean() - por_zona.query("t==0").y.mean()

res = {nivel: np.array([experimento(nivel, s) for s in range(300)]) for nivel in ["usuario", "zona"]}
print(f"Efecto de lanzar a todos (verdad): {TAU - GAMMA:+.2f}")
print(pd.DataFrame({k: {"media estimada": v.mean(), "desviación típica": v.std()} for k, v in res.items()}).T.rename(index={"usuario": "aleatorizar por usuario", "zona": "aleatorizar por zona"}))

**Lee la tabla.** Aleatorizando por usuario se mide el efecto *directo* (≈ 1,0) y se ignora la competencia entre usuarios de la misma zona; el lanzamiento completo solo dará ≈ 0,2. Aleatorizar por zona (*cluster*) recupera el efecto real a cambio de menos precisión (menos unidades independientes). Variantes: *switchback* (alternar por tiempo) y aleatorización por grafo.

## DS2.7 · *Bandits*: explorar y explotar a la vez

Un A/B reparte tráfico fijo aunque una variante sea claramente peor. Un **bandit** reparte dinámicamente: dirige más sesiones a lo que parece mejor, reduciendo el *regret* (lo que pierdes por no mostrar siempre la mejor). **ε-greedy** explora al azar una fracción ε; **Thompson sampling** muestrea de la distribución posterior de cada variante y elige la mejor muestra: explora más donde hay incertidumbre.

Compromiso: aprendes menos sobre las variantes que no se muestran (los intervalos de confianza son más difíciles) y asumes una métrica estable. Va muy bien para optimizar titulares, banners o recomendaciones de vida corta; un A/B sigue siendo mejor para decisiones de producto con inferencia.

In [ ]:
PROBS = np.array([0.100, 0.110, 0.115, 0.130])      # conversión real de 4 variantes del carrito (A = actual); el mejor es el 4.º
T, RUNS = 10_000, 100

def correr(estrategia, probs=PROBS, T=T, runs=RUNS, semilla=0, **kw):
    """Devuelve el arrepentimiento (regret) esperado acumulado medio y la proporción de veces que se eligió el mejor brazo."""
    r = np.random.default_rng(semilla)
    K = len(probs)
    s = np.zeros((runs, K)); n = np.zeros((runs, K))
    regret = np.zeros(T); eligio_mejor = np.zeros(T)
    mejor = probs.argmax(); filas = np.arange(runs)
    for t in range(T):
        if estrategia == "uniforme":                       # A/B/n clásico: reparto fijo
            a = np.full(runs, t % K)
        elif estrategia == "epsilon":
            eps = kw["eps"]
            medias = np.where(n > 0, s / np.maximum(n, 1), 1.0)       # brazos sin probar: optimistas
            a = np.where(r.random(runs) < eps, r.integers(0, K, runs), medias.argmax(1))
        elif estrategia == "thompson":
            a = r.beta(1 + s, 1 + n - s).argmax(1)             # una muestra de cada posterior; elegimos la mayor
        exito = r.random(runs) < probs[a]
        s[filas, a] += exito; n[filas, a] += 1
        regret[t] = (probs[mejor] - probs[a]).mean(); eligio_mejor[t] = (a == mejor).mean()
    return np.cumsum(regret), eligio_mejor

curvas = {"A/B/n uniforme": correr("uniforme"), "ε-greedy (ε = 0,10)": correr("epsilon", eps=0.10),
          "ε-greedy (ε = 0,02)": correr("epsilon", eps=0.02), "Thompson sampling": correr("thompson")}
fig, ax = plt.subplots(figsize=(7.5, 4))
for nombre, (reg, _) in curvas.items():
    ax.plot(reg, label=nombre)
ax.set(xlabel="sesiones", ylabel="regret acumulado (conversiones perdidas)", title="Regret frente a siempre mostrar la mejor variante"); ax.legend(); plt.show()
print(pd.DataFrame({k: {"regret final": v[0][-1], "% de sesiones con la mejor variante": v[1].mean() * 100,
                        "% en las últimas 1 000": v[1][-1000:].mean() * 100} for k, v in curvas.items()}).T.round(1))

**Lee la tabla.** El A/B/n uniforme acumula regret linealmente (nunca deja de mostrar variantes malas). Thompson lo reduce con fuerza y acaba mostrando la mejor variante casi siempre. Un ε fijo sigue desperdiciando una fracción ε de por vida.

## DS2.8 · Caso integrado: informe del rediseño del carrito

Juntamos todo en el orden en el que lo harías en un análisis real: **(1) calidad del experimento (SRM) → (2) métrica principal con CUPED → (3) secundarias corregidas → (4) guardarraíles → decisión.**

In [ ]:
def generar_carrito(n=40_000, semilla=43, srm_roto=False):
    """Experimento del rediseño del carrito. Efecto real sobre el gasto: +0,8 €; sobre la conversión: +4 % relativo;
    el tiempo de carga empeora 12 ms. Se miden también 10 métricas secundarias SIN efecto."""
    r = np.random.default_rng(semilla)
    nivel = r.lognormal(3.2, 0.7, 2 * n)
    g = np.r_[np.zeros(n), np.ones(n)].astype(int)
    x_pre = nivel + r.normal(0, 8, 2 * n)
    y = nivel + 0.8 * g + r.normal(0, 8, 2 * n)
    conv = r.random(2 * n) < 0.10 * (1 + 0.04 * g)
    carga = r.gamma(8, 100, 2 * n) + 12 * g
    sec = {f"metrica_sec_{i}": r.normal(0, 1, 2 * n) for i in range(1, 11)}
    d = pd.DataFrame({"grupo": g, "gasto_pre": x_pre, "gasto": y, "compra": conv.astype(int), "carga_ms": carga, **sec})
    if srm_roto:
        d = d.drop(d.index[(d.grupo == 1) & (d.gasto_pre < np.quantile(d.gasto_pre, 0.2)) & (r.random(len(d)) < 0.5)])
    return d

df = generar_carrito()
print("1) SRM:", f"{(df.grupo == 1).sum():,} / {(df.grupo == 0).sum():,}", "-> p =", round(srm((df.grupo == 1).sum(), (df.grupo == 0).sum()), 3))
t, c = df[df.grupo == 1], df[df.grupo == 0]

print("\n2) Métrica principal (gasto por usuario), con y sin CUPED")
theta = np.cov(df.gasto, df.gasto_pre)[0, 1] / df.gasto_pre.var(ddof=1)
df["gasto_cuped"] = df.gasto - theta * (df.gasto_pre - df.gasto_pre.mean())
for etiqueta, col in [("sin CUPED", "gasto"), ("con CUPED", "gasto_cuped")]:
    d, se, ic, p = dif_medias(df.loc[df.grupo == 1, col], df.loc[df.grupo == 0, col])
    print(f"   {etiqueta}: {d:+.2f} € (IC95% [{ic[0]:+.2f}, {ic[1]:+.2f}], p = {p:.4f})")

print("\n3) Secundarias: conversión + 10 métricas, corregidas con Holm")
nombres = ["compra"] + [f"metrica_sec_{i}" for i in range(1, 11)]
pv = [dif_medias(t[m], c[m])[3] for m in nombres]
rech, p_aj, _, _ = multipletests(pv, alpha=0.05, method="holm")
print(pd.DataFrame({"métrica": nombres, "p crudo": pv, "p ajustado (Holm)": p_aj, "significativa": rech}).to_string(index=False))

print("\n4) Guardarraíl: tiempo de carga (margen tolerado 30 ms)")
d, lim, ok = guardarrail(t.carga_ms, c.carga_ms, margen=30)
print(f"   dif = {d:+.1f} ms, límite superior 95 % = {lim:.1f} ms -> {'PASA' if ok else 'FALLA'}")

**Decisión redactada.** El SRM es limpio (40 000 / 40 000). El gasto por usuario sube ≈ 0,9 € y CUPED estrecha el intervalo a menos de la mitad (de ±0,37 a ±0,16). La conversión no muestra diferencia concluyente: el experimento no tenía potencia para un +4 % relativo (comprueba con : harían falta unos 90 000 por grupo), así que **no** se afirma que mejore. De las 10 métricas sin efecto, una sale con p crudo = 0,014 y Holm la descarta (p ajustado = 0,16): sin corrección habríamos «descubierto» un efecto falso. El tiempo de carga empeora ≈ 11 ms, dentro del margen. Recomendación: **lanzar**, con seguimiento de 4 semanas para descartar efecto novedad. Fíjate en que la conclusión se apoya en un orden de comprobaciones, no solo en un p-valor.

## Ejercicios

Las funciones y los datos de las secciones anteriores están disponibles.

1. **Tamaño de muestra con CUPED.** Para la métrica «gasto por sesión» (σ = 40 €) quieres detectar +1 €. Calcula el tamaño por grupo (α = 0,05, potencia 0,8) sin CUPED y con CUPED si ρ(pre, post) = 0,7. Comprueba que la razón entre ambos es ≈ 1 − ρ².
2. **SRM.** Escribe una función `revisar_srm(n_t, n_c, prop_t)` que devuelva un diagnóstico («OK» / «SRM») con umbral p < 0,001 y pruébala con: 10 400 vs 10 000 (reparto 50/50); 5 300 vs 4 700 (reparto 50/50) y 3 400 vs 6 600 (reparto diseñado 33/67). Explica en un comentario por qué se usa un umbral tan estricto.
3. **Mirar 5 veces.** Calibra por simulación el límite de Pocock para K = 5 miradas (reutiliza `trayectorias_z`). Mide después la tasa de falsos positivos de «mirar 5 veces con 1,96» y la de Pocock con 20 000 experimentos A/A.
4. **Holm y Benjamini-Hochberg a mano.** Con los p-valores `[0.001, 0.008, 0.012, 0.03, 0.04, 0.2, 0.5, 0.9]` implementa Holm y BH (α = 0,05) sin `multipletests` y comprueba que coinciden con la librería. ¿Qué hipótesis se rechazan con cada método?
5. **Bandits.** Con `correr("epsilon", eps=...)` prueba ε ∈ {0,01; 0,05; 0,2; 0,5} y compara el regret final con el de Thompson. ¿Por qué un ε grande se parece al A/B uniforme y uno muy pequeño puede quedarse atascado?


In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
